# 05 — Deep Learning Models (TensorFlow)

Train CNN, LSTM, and CNN-LSTM models directly on raw EEG epochs (no manual feature engineering).

**What this notebook covers:**
- Building 1D-CNN, LSTM, and CNN-LSTM architectures
- Training with early stopping and LR scheduling
- Plotting training curves
- Confusion matrices and ROC curves
- Comparing deep vs. classical model performance

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

from src.models.deep_learning import build_cnn, build_lstm, build_cnn_lstm, EEGDeepClassifier
from src.utils.evaluation import (
    plot_confusion_matrix,
    plot_training_history,
    plot_roc_curves,
    compare_models,
)

print(f'TF version: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')
%matplotlib inline

TASK = 'sleep'
CLASS_NAMES = ['Wake', 'N1', 'N2', 'N3', 'REM']
SFREQ = 100

## 1. Load Epoch Data

In [ ]:
epochs = np.load(f'../data/processed/{TASK}_data.npy').astype(np.float32)
labels = np.load(f'../data/processed/{TASK}_labels.npy').astype(int)

n_classes    = len(np.unique(labels))
input_shape  = epochs.shape[1:]   # (n_channels, n_samples)

print(f'Epochs : {epochs.shape}')
print(f'Labels : {labels.shape}  |  Classes: {n_classes}')
print(f'Input shape for models: {input_shape}')

X_train, X_test, y_train, y_test = train_test_split(
    epochs, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 2. CNN Model

In [ ]:
cnn_model = build_cnn(input_shape=input_shape, n_classes=n_classes)
cnn_model.summary()

cnn_clf = EEGDeepClassifier(cnn_model, n_classes=n_classes, task='multiclass')
cnn_history = cnn_clf.train(
    X_train, y_train,
    X_val=(X_val, y_val),
    epochs=60,
    batch_size=64,
    patience=12,
    save_best_path='../models/deep/sleep_cnn_best.keras'
)

In [ ]:
plot_training_history(cnn_history)

## 3. CNN-LSTM Model (Recommended for Sleep Staging)

In [ ]:
cnnlstm_model = build_cnn_lstm(
    input_shape=input_shape, n_classes=n_classes,
    cnn_filters=[64, 128], lstm_units=128
)
cnnlstm_clf = EEGDeepClassifier(cnnlstm_model, n_classes=n_classes, task='multiclass')
cnnlstm_history = cnnlstm_clf.train(
    X_train, y_train,
    X_val=(X_val, y_val),
    epochs=80,
    batch_size=64,
    patience=15,
    save_best_path='../models/deep/sleep_cnn_lstm_best.keras'
)

## 4. Evaluate Both Models

In [ ]:
print('=== CNN ===')
cnn_results = cnn_clf.evaluate(X_test, y_test)

print('\n=== CNN-LSTM ===')
cnnlstm_results = cnnlstm_clf.evaluate(X_test, y_test)

compare_models({'CNN': cnn_results, 'CNN-LSTM': cnnlstm_results}, metric='accuracy')

## 5. Confusion Matrix — Best Model

In [ ]:
best_results = cnnlstm_results if cnnlstm_results['accuracy'] > cnn_results['accuracy'] else cnn_results

plot_confusion_matrix(
    best_results['confusion_matrix'],
    class_names=CLASS_NAMES,
    title='CNN-LSTM Sleep Stage Confusion Matrix',
    normalize=True
)

## 6. ROC Curves

In [ ]:
y_prob = cnnlstm_model.predict(X_test, verbose=0)
plot_roc_curves(y_test, y_prob, CLASS_NAMES)